In [1]:
# environment setup
# %conda install -q transformers datasets accelerate sentencepiece protobuf
# %conda install -q torch

In [2]:
import csv
import os
import random
import re
from pathlib import Path
import torch

from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

os.environ["WANDB_DISABLED"] = "true"

MODEL_NAME = "google/flan-t5-large"
OUTPUT_DIR = "outputs/flan-t5-large-idiom-combined"

SEED = 42
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128
TRAIN_EPOCHS = 2
LEARNING_RATE = 1e-4
BATCH_SIZE = 2

# Keep this small for quick smoke tests. Set to None to train on every CSV row.
MAX_REAL_ROWS = None
INCLUDE_GENERATION_TASK = True
TEST_FRACTION = 0.2
SAMPLE_EXAMPLES_PER_TASK = 2
SANITY_CHECK_EXAMPLES = 2

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Ti


In [3]:
# data loading

CSV_CANDIDATES = [
    Path("dataset/idioms_dataset.csv"),
    # Path("data/flute_idioms.csv"),
    # Path("flute_idioms.csv"),
]

data_csv = next((path for path in CSV_CANDIDATES if path.exists()), None)


def mask_idiom(text, idiom, mask="[IDIOM]"):
    escaped_idiom = re.escape(idiom.strip())
    return re.sub(rf"(?<!\w){escaped_idiom}(?!\w)", mask, text, flags=re.IGNORECASE)


def make_interpretation_example(row):
    idiom = row["idiom"].strip()
    sentence = row["example"].strip()
    meaning = row["explanation_correct"].strip()
    prompt = (
        f"Idiom: {idiom}\n"
        f"Sentence: {sentence}"
    )
    return {
        "task": "interpretation",
        "input": prompt,
        "target": meaning,
    }


def make_generation_example(row):
    idiom = row["idiom"].strip()
    sentence = row["example"].strip()
    meaning = row["explanation_correct"].strip()
    scenario = row.get("correct_substitution", "").strip() or sentence
    masked_meaning = mask_idiom(meaning, idiom)
    masked_scenario = mask_idiom(scenario, idiom)
    prompt = (
        f"Meaning: {masked_meaning}\n"
        f"Scenario: {masked_scenario}"
    )
    return {
        "task": "generation",
        "input": prompt,
        "target": f"Idiom: {idiom}\nSentence: {sentence}",
    }


if data_csv is None:
    expected_paths = ", ".join(str(path) for path in CSV_CANDIDATES)
    raise FileNotFoundError(f"Could not find formatted FLUTE CSV. Expected one of: {expected_paths}")

with data_csv.open("r", encoding="utf-8-sig", newline="") as csv_file:
    rows = list(csv.DictReader(csv_file))

required_columns = {"idiom", "example", "explanation_correct"}
missing_columns = required_columns - set(rows[0].keys() if rows else [])
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")

rows = [row for row in rows if all(row.get(column, "").strip() for column in required_columns)]
if MAX_REAL_ROWS is not None:
    rows = rows[:MAX_REAL_ROWS]

if len(rows) < 2:
    raise ValueError("Need at least 2 usable CSV rows to make a train/test split.")

split_rows = rows.copy()
random.Random(SEED).shuffle(split_rows)

test_row_count = max(1, round(len(split_rows) * TEST_FRACTION))
test_row_count = min(test_row_count, len(split_rows) - 1)
test_rows = split_rows[:test_row_count]
train_rows = split_rows[test_row_count:]


def build_task_examples(source_rows):
    examples = []
    for row in source_rows:
        examples.append(make_interpretation_example(row))
        if INCLUDE_GENERATION_TASK:
            examples.append(make_generation_example(row))
    return examples


train_examples = build_task_examples(train_rows)
test_examples = build_task_examples(test_rows)

print(f"Loaded {len(rows)} usable CSV rows from {data_csv}")
print(f"Split into {len(train_rows)} training rows and {len(test_rows)} held-out test rows")
print(f"Built {len(train_examples)} training examples and {len(test_examples)} held-out test examples")

train_dataset = Dataset.from_list(train_examples)
test_dataset = Dataset.from_list(test_examples)
test_interpretation_examples = [example for example in test_examples if example["task"] == "interpretation"]
test_generation_examples = [example for example in test_examples if example["task"] == "generation"]

preview_interpretation_examples = test_interpretation_examples[:SAMPLE_EXAMPLES_PER_TASK]
preview_generation_examples = test_generation_examples[:SAMPLE_EXAMPLES_PER_TASK]
train_dataset

Loaded 1515 usable CSV rows from dataset\idioms_dataset.csv
Split into 1212 training rows and 303 held-out test rows
Built 2424 training examples and 606 held-out test examples


Dataset({
    features: ['task', 'input', 'target'],
    num_rows: 2424
})

In [4]:
# load FLAN-T5-Large
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(f"Loaded {MODEL_NAME}")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded google/flan-t5-large


In [5]:
# sampling

def generate_text(prompt, max_new_tokens=96, do_sample=False, temperature=0.7):
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "num_beams": 4,
        "do_sample": do_sample,
    }
    if do_sample:
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_p"] = 0.9

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(**encoded, **generation_kwargs)

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def sample_examples(title, examples):
    print("=" * 80)
    print(title)
    print("=" * 80)
    if not examples:
        print("No examples available for this task.")
        return
    for idx, example in enumerate(examples, start=1):
        print(f"\nExample {idx}: {example['task']}")
        print("-" * 80)
        print("PROMPT:")
        print(example["input"])
        print("\nMODEL OUTPUT:")
        print(generate_text(example["input"]))
        print("\nTARGET:")
        print(example["target"])


sample_examples("Before fine-tuning: interpretation", preview_interpretation_examples)

Before fine-tuning: interpretation

Example 1: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.

MODEL OUTPUT:
J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.

TARGET:
In your face means brashly confrontational, which J recognises is unnecessary.

Example 2: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: same difference
Sentence: With a beard, without a beard, same difference.

MODEL OUTPUT:
With a beard, without a beard, same difference.

TARGET:
Same difference means the distinction makes no practical difference.


In [6]:
# sample generation examples before fine-tuning
sample_examples("Before fine-tuning: generation", preview_generation_examples)

Before fine-tuning: generation

Example 1: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: [IDIOM] means brashly confrontational, which J recognises is unnecessary.
Scenario: J is angry, but he realises you haven't got to get totally aggressive and confrontational to get a serious message across.

MODEL OUTPUT:
[IDIOM]

TARGET:
Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.

Example 2: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: [IDIOM] means the distinction makes no practical difference.
Scenario: With a beard, without a beard, it amounts to the same thing.

MODEL OUTPUT:
[IDIOM]

TARGET:
Idiom: same difference
Sentence: With a beard, without a beard, same difference.


In [7]:
#tokenize data

def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

tokenized_train

Map:   0%|          | 0/2424 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2424
})

In [8]:
# sanity-check labels and loss before training
# Use a tiny debug batch here. Running this over the full training set can cause CUDA OOM.
debug_count = min(SANITY_CHECK_EXAMPLES, len(tokenized_train))
debug_indices = list(range(debug_count))
debug_batch = data_collator([tokenized_train[i] for i in debug_indices])
debug_batch = {key: value.to(model.device) for key, value in debug_batch.items()}

non_padding_target_tokens = (debug_batch["labels"] != -100).sum().item()
print(f"Non-padding target tokens: {non_padding_target_tokens}")

for idx in debug_indices:
    print(f"\nDecoded target {idx + 1}:")
    print(tokenizer.decode(tokenized_train[idx]["labels"], skip_special_tokens=True))

model.eval()
with torch.no_grad():
    initial_loss = model(**debug_batch).loss.item()

print(f"\nInitial loss before fine-tuning: {initial_loss:.4f}")
assert non_padding_target_tokens > 0, "No target tokens found. Check preprocessing."
assert torch.isfinite(torch.tensor(initial_loss)), "Initial loss is not finite."
assert initial_loss > 0, "Initial loss should usually be nonzero before training."


Non-padding target tokens: 62

Decoded target 1:
To hedge your bets means to protect yourself by not committing to a single outcome, which the information supports.

Decoded target 2:
Idiom: hedge your bets Sentence: ANDREW ARROWSMITH has the information to help you hedge your bets.

Initial loss before fine-tuning: 2.1018


In [9]:
# fine-tuning
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=TRAIN_EPOCHS,
    weight_decay=0.0,
    logging_steps=1000,
    save_strategy="no",
    report_to="none",
    # Keep fp16 off for this tiny demo. In this environment it caused 0.0 loss and nan gradients.
    fp16=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss
1000,1.114076
2000,0.760045


TrainOutput(global_step=2424, training_loss=0.8964107752633174, metrics={'train_runtime': 3187.845, 'train_samples_per_second': 1.521, 'train_steps_per_second': 0.76, 'total_flos': 1077857143013376.0, 'train_loss': 0.8964107752633174, 'epoch': 2.0})

In [19]:
# sample interpretation examples after fine-tuning
sample_examples("After fine-tuning: interpretation", preview_interpretation_examples)

After fine-tuning: interpretation

Example 1: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.

MODEL OUTPUT:
To get in your face means to act aggressively and aggressively, which is what J is angry about.

TARGET:
In your face means brashly confrontational, which J recognises is unnecessary.

Example 2: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: same difference
Sentence: With a beard, without a beard, same difference.

MODEL OUTPUT:
The same difference means that something is different from something else, and in this context it is saying that with a beard or without a beard there is no difference.

TARGET:
Same difference means the distinction makes no practical difference.


In [20]:
# sample generation examples after fine-tuning
sample_examples("After fine-tuning: generation", preview_generation_examples)

After fine-tuning: generation

Example 1: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: [IDIOM] means brashly confrontational, which J recognises is unnecessary.
Scenario: J is angry, but he realises you haven't got to get totally aggressive and confrontational to get a serious message across.

MODEL OUTPUT:
Idiom: rough and ready Sentence: J is angry, but he realises you haven't got to get rough and ready to get a serious message across.

TARGET:
Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.

Example 2: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: [IDIOM] means the distinction makes no practical difference.
Scenario: With a beard, without a beard, it amounts to the same thing.

MODEL OUTPUT:
Idiom: with or without a beard Sentence: With a beard, without a beard, it amounts

In [21]:
# save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved fine-tuned model and tokenizer to: {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model and tokenizer to: outputs/flan-t5-large-idiom-combined


In [22]:
# environment setup for BERTScore evaluation
# %conda install -c conda-forge bert_score
from bert_score import score

In [29]:
# Evaluate the fine-tuned model with BERTScore on the held-out test set.
# Compare each model prediction against that same example's target text.

predictions = []
references = []

for example in test_interpretation_examples:
    prompt = example["input"]
    target = example["target"]

    pred_text = generate_text(prompt)

    predictions.append(pred_text)
    references.append(target)

# Compute BERTScore across the whole test set
P, R, F1 = score(
    predictions, 
    references, 
    model_type=model.config._name_or_path,
    num_layers=24,
    # device=device
    )

bertscore_summary = {
    "precision": float(P.mean().item()),
    "recall": float(R.mean().item()),
    "f1": float(F1.mean().item()),
}

print(f"BERTScore on test set:")
print(f"  Precision: {bertscore_summary['precision']:.4f}")
print(f"  Recall:    {bertscore_summary['recall']:.4f}")
print(f"  F1:        {bertscore_summary['f1']:.4f}")

# Optional: show a few example predictions
print("\nSample predictions vs targets:")
for i in range(min(3, len(predictions))):
    print(f"\nExample {i + 1}")
    print("INPUT:   ", test_interpretation_examples[i]["input"])
    print("PRED:    ", predictions[i])
    print("TARGET:  ", references[i])

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

[transformers] T5EncoderModel LOAD REPORT from: google/flan-t5-large
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore on test set:
  Precision: 0.6621
  Recall:    0.6664
  F1:        0.6617

Sample predictions vs targets:

Example 1
INPUT:    Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.
PRED:     To get in your face means to act aggressively and aggressively, which is what J is angry about.
TARGET:   In your face means brashly confrontational, which J recognises is unnecessary.

Example 2
INPUT:    Idiom: same difference
Sentence: With a beard, without a beard, same difference.
PRED:     The same difference means that something is different from something else, and in this context it is saying that with a beard or without a beard there is no difference.
TARGET:   Same difference means the distinction makes no practical difference.

Example 3
INPUT:    Idiom: turn over a new leaf
Sentence: There is no indication that Hollywood is turning over a new leaf, free of bloodstains.
PRED:     To turn over a new

In [33]:
custom_generation_example = [
    {
        'task': 'generation',
        'input': (
            'Meaning: The idiom [IDIOM] means to do very well, such as turning in a great class project.\n'
            'Scenario: Alex, Jonathan, and Marcus performed exceptionally well on their NLP project last week and everyone was very impressed.'
        ),
        'target': ''
    }]
custom_interpretation_example = [
    {
        'task': 'interpretation',
        'input': (
            'Idiom: Its all tokens to me.\n'
            'Sentence: Jonathan, Alex, and Marcus had much difficulty trying to interpret the output of their trained AI model. Everything it'
            'produced was non-sensical. "Its all tokens to me", Marcus said, confusedly.'
        ),
        'target': ''
    }]

In [34]:
sample_examples('Custom Idiom', custom_generation_example)

Custom Idiom

Example 1: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: The idiom [IDIOM] means to do very well, such as turning in a great class project.
Scenario: Alex, Jonathan, and Marcus performed exceptionally well on their NLP project last week and everyone was very impressed.

MODEL OUTPUT:
Idiom: out of the blue Sentence: Alex, Jonathan, and Marcus out of the blue on their NLP project last week and everyone was very impressed.

TARGET:



In [35]:
sample_examples('Custom Idiom', custom_interpretation_example)

Custom Idiom

Example 1: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: Its all tokens to me.
Sentence: Jonathan, Alex, and Marcus had much difficulty trying to interpret the output of their trained AI model. Everything itproduced was non-sensical. "Its all tokens to me", Marcus said, confusedly.

MODEL OUTPUT:
The idiom "it's all tokens to me" means that Marcus is confused about what the model is telling him.

TARGET:

